In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.optimize import least_squares
from scipy.special import spherical_jn as jn
import emcee
import os
from multiprocessing import Pool
import corner

import utils.lib.nn_studio as nn_studio
import utils.lib.chiral_potential as chiral_potential
import utils.lib.granada_phases as granada
import utils.lib.auxiliary as aux
import utils.lib.lec_values as lec_values
from utils.lib.constants import *

import time
import datetime

In [2]:
%matplotlib widget
plt.close('all')

# Inferenza bayesiana NLO

Possiamo disaccoppiare i parametri su cui facciamo inferenza grazie alle simmetrie dell'interazione nucleare, in particolare avremo un blocco 2D e uno 3D.

Possiamo fare la separazione grazie alle seguenti regole di conservazione della Hamiltoniaa nucleare:
- spin totale $S$
- isospin totale $T$
- momento angolare totale $\vec{J} = \vec{L} + \vec{S}$
- parità $\pi = (-1)^L$

inoltre, trattandosi di un sistema di due fermioni identici, il principio di Pauli impone che la funzione d'onda totale sia completamente antisimmetrica, richiedendo che $L+S+T$ sia un numero dispari. 

In [3]:
T_Lab = granada.Tlabs 

shift_1S0 = granada.delta_1S0
shift_1S0_error = granada.delta_1S0_errors

shift_3S1 = granada.delta_3S1
shift_3S1_error = granada.delta_3S1_errors

shift_3D1 = granada.delta_3D1
shift_3D1_error = granada.delta_3D1_errors

shift_mix = granada.delta_3E1
shift_mix_error = granada.delta_3E1_errors

In [ ]:
nn = nn_studio.nn_studio(jmin=0, jmax=1, tzmin=0, tzmax=0, Np=30, mesh_type="gauleg_infinite")
nn.Tlabs = T_Lab
nn.V = chiral_potential.two_nucleon_potential("NLO",Lambda=500.0)

_, channel_1S0 = nn.lookup_channel_idx(l=0,ll=0,s=0,j=0) 
_, channel_3S1 = nn.lookup_channel_idx(l=0,ll=2,s=1,j=1) 

## Blocco $^1S_0$

Consideriamo lo stato in onda S di singoletto: $L=0$ e $S=0$, che implica $J=0$. Per il principio di Pauli, affinché $L+S+T$ sia dispari ($0+0+T$), l'isospin deve essere necessariamente $T=1$. Lo stato quantistico completo è quindi:
$$| \psi \rangle = | J=0 , L=0 , S=0 \rangle | T=1 \rangle$$
L'Hamiltoniana nucleare potrebbe mescolare stati con diverso $L$, a patto di conservare $\pi$, $S$, $J$ e $T$. Tuttavia, non esiste alcun altro valore di momento angolare orbitale $L$ che, combinato con spin $S=0$, possa dare $J=0$. Di conseguenza, gli elementi di matrice di transizione verso altri stati sono rigorosamente nulli:
$$\langle J=0 , L=0 , S=0 | \mathcal{H} | \text{qualsiasi altro stato} \rangle = 0$$
L'onda $^1S_0$ è un canale isolato. Le osservabili ad essa associate dipendono esclusivamente dai parametri di contatto e di correzione ai momenti specifici per questo stato, ovvero $C_{^1S_0}$ e $D_{^1S_0}$

In [ ]:
# Inferenza su due soli parametri: C_1S0 e C_3S1.
# Gli altri LEC NLO sono fissati ai valori nominali.
def ln_prior_NLO(parameters):
    if len(parameters) != 2:
        return -np.inf

    C_1S0, D_1S0 = parameters

    avg_C1S0 = 0.0
    avg_D1S0 = 0.0
    sigma_C1S0 = 5.0
    sigma_D1S0 = 5.0

    ln_C1S0 = -0.5 * (np.log(2 * np.pi * sigma_C1S0**2) + ((C_1S0 - avg_C1S0) / sigma_C1S0) ** 2)
    ln_D1S0 = -0.5 * (np.log(2 * np.pi * sigma_D1S0**2) + ((D_1S0 - avg_D1S0) / sigma_D1S0) ** 2)

    # se un parametro e fuori di 3 sigma, il prior e -inf
    if abs(C_1S0 - avg_C1S0) > 3 * sigma_C1S0:
        return -np.inf
    if abs(D_1S0 - avg_D1S0) > 3 * sigma_D1S0:
        return -np.inf

    return ln_C1S0 + ln_D1S0


def ln_likelihood_NLO(parameters):
    NLO_lecs = {}
    NLO_lecs["C_1S0"] = parameters[0]
    NLO_lecs["D_1S0"] = parameters[1]

    # Parametri non inferiti: fissati ai valori nominali NLO
    NLO_lecs["C_3S1"] = lec_values.nlo_lecs["C_3S1"]
    NLO_lecs["D_3S1"] = lec_values.nlo_lecs["D_3S1"]
    NLO_lecs["D_3S1-3D1"] = lec_values.nlo_lecs["D_3S1-3D1"]
    NLO_lecs["D_1P1"] = lec_values.nlo_lecs["D_1P1"]
    NLO_lecs["D_3P0"] = lec_values.nlo_lecs["D_3P0"]
    NLO_lecs["D_3P1"] = lec_values.nlo_lecs["D_3P1"]
    NLO_lecs["D_3P2"] = lec_values.nlo_lecs["D_3P2"]

    nn.lecs = NLO_lecs

    nn.compute_Tmtx(channel_1S0, verbose=False)
    teo_1S0 = nn.phase_shifts[0]
    
    ln_L_1S0 = -0.5 * np.sum(np.log(2 * np.pi * err_1S0**2) + ((exp_1S0 - teo_1S0) / err_1S0) ** 2)

    return ln_L_1S0


def ln_posterior_NLO(parameters):
    ln_pr = ln_prior_NLO(parameters)
    if not np.isfinite(ln_pr):
        return -np.inf

    ln_post = ln_pr
    ln_post += ln_likelihood_NLO(parameters)
    return ln_post


In [6]:
# MCMC
ndim = 2
nwalkers = 32
nsteps = 100

rng = np.random.default_rng(42)  # riproducibilita
initial_guess = np.array([lec_values.nlo_lecs["C_1S0"], lec_values.nlo_lecs["C_3S1"]])
initial_pos = initial_guess + 0.05 * rng.standard_normal((nwalkers, ndim))

ncpu = os.cpu_count()

if __name__ == "__main__":
    with Pool(processes=ncpu) as pool:
        sampler = emcee.EnsembleSampler(
            nwalkers,
            ndim,
            ln_posterior_NLO,
            args=(),
            pool=pool,
        )
        print(f"Eseguendo su {ncpu} core...")
        sampler.run_mcmc(initial_pos, nsteps, progress=True)

    print("\nFinito! :)\n")

Eseguendo su 12 core...


  1%|          | 1/100 [00:19<32:31, 19.72s/it]

emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:

  1%|          | 1/100 [00:30<49:59, 30.30s/it]

emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:

emcee: Exception while calling your likelihood function:

  params:  params:



  

KeyboardInterrupt: 

In [ ]:
full_chain = sampler.get_chain()

fig, axes = plt.subplots(2, figsize=(10, 7), sharex=True)
labels = ["C_1S0", "C_3S1"]

for i in range(ndim):
    ax = axes[i]
    for walker in range(nwalkers):
        ax.plot(full_chain[:, walker, i], alpha=0.5)
    ax.set_ylabel(labels[i])
axes[-1].set_xlabel("Step")
plt.suptitle("Tracce dei cammini dei walker")
plt.tight_layout()
plt.show()

In [ ]:
corner.corner(sampler.get_chain(discard=0, thin=1, flat=True), labels=labels, show_titles=True)
plt.suptitle("Corner plot dei campioni posteriors")
plt.tight_layout()
plt.show()

In [ ]:
tau = sampler.get_autocorr_time()

## Blocco $^3S_1 - ^3D_1$

Consideriamo ora lo stato del deuterio, caratterizzato da spin $S=1$ e momento angolare totale $J=1$. Affinché la somma $L+S+T$ sia dispari, l'isospin deve essere $T=0$.Quali valori di $L$ possono generare $J=1$ accoppiandosi con $S=1$?$L=0$ (onda S, $\pi = +1$)$L=1$ (onda P, $\pi = -1$)$L=2$ (onda D, $\pi = +1$)Poiché l'interazione forte conserva la parità, lo stato in onda P non può mescolarsi con gli altri. Rimangono quindi le onde S e D, che hanno la stessa parità. L'operatore tensoriale $S_{12}$ presente nell'Hamiltoniana nucleare ha elementi di matrice non nulli tra questi due stati, generando un mescolamento (mixing):
$$\langle J=1 , L=0 , S=1  | \mathcal{H} | J=0 , L=2 , S=0  \rangle \neq 0$$
Il sistema non è più descritto da un singolo ket, ma da una matrice $2 \times 2$ nello spazio $\left\{ | ^3S_1 \rangle, | ^3D_1 \rangle \right\}$. Lo scattering e le proprietà dello stato legato (come il momento di quadrupolo) in questo blocco dipendono esclusivamente e congiuntamente dalle costanti $C_{^3S_1}$, $D_{^3S_1}$ e dal termine di accoppiamento tensoriale $D_{^3E_1}$.

In [ ]:
# Parametri attivi: C_3S1, D_3S1, D_mix (nel dizionario "D_3S1-3D1")

def ln_prior_NLO_3S1(parameters):
    if len(parameters) != 3:
        return -np.inf

    C_3S1, D_3S1, D_mix = parameters

    avg_C3S1 = 0.0
    avg_D3S1 = 0.0
    avg_Dmix = 0.0

    sigma_C3S1 = 5.0
    sigma_D3S1 = 5.0
    sigma_Dmix = 5.0

    ln_C3S1 = -0.5 * (np.log(2 * np.pi * sigma_C3S1**2) + ((C_3S1 - avg_C3S1) / sigma_C3S1) ** 2)
    ln_D3S1 = -0.5 * (np.log(2 * np.pi * sigma_D3S1**2) + ((D_3S1 - avg_D3S1) / sigma_D3S1) ** 2)
    ln_Dmix = -0.5 * (np.log(2 * np.pi * sigma_Dmix**2) + ((D_mix - avg_Dmix) / sigma_Dmix) ** 2)

    # se un parametro e fuori di 3 sigma, il prior e -inf
    if abs(C_3S1 - avg_C3S1) > 3 * sigma_C3S1:
        return -np.inf
    if abs(D_3S1 - avg_D3S1) > 3 * sigma_D3S1:
        return -np.inf
    if abs(D_mix - avg_Dmix) > 3 * sigma_Dmix:
        return -np.inf

    return ln_C3S1 + ln_D3S1 + ln_Dmix


def ln_likelihood_NLO_3S1(parameters, nnstudio, chn_3S1, exp_3S1, err_3S1, exp_3D1, err_3D1, exp_mix, err_mix):
    from scipy.interpolate import CubicSpline

    NLO_lecs = {}

    # Inseriamo i 3 parametri che stiamo campionando
    NLO_lecs["C_3S1"] = parameters[0]
    NLO_lecs["D_3S1"] = parameters[1]
    NLO_lecs["D_3S1-3D1"] = parameters[2]

    # I parametri degli altri canali sono fissati ai valori nominali NLO
    NLO_lecs["C_1S0"] = lec_values.nlo_lecs["C_1S0"]
    NLO_lecs["D_1S0"] = lec_values.nlo_lecs["D_1S0"]
    NLO_lecs["D_1P1"] = lec_values.nlo_lecs["D_1P1"]
    NLO_lecs["D_3P0"] = lec_values.nlo_lecs["D_3P0"]
    NLO_lecs["D_3P1"] = lec_values.nlo_lecs["D_3P1"]
    NLO_lecs["D_3P2"] = lec_values.nlo_lecs["D_3P2"]

    nnstudio.lecs = NLO_lecs

    # 1) Likelihood sui phase shifts del canale accoppiato 3S1-3D1
    nnstudio.compute_Tmtx(chn_3S1, verbose=False)
    teo_3S1 = nnstudio.phase_shifts[0][:, 0]
    teo_mix = nnstudio.phase_shifts[0][:, 1]
    teo_3D1 = nnstudio.phase_shifts[0][:, 2]

    ln_L_3S1 = -0.5 * np.sum(np.log(2 * np.pi * err_3S1**2) + ((exp_3S1 - teo_3S1) / err_3S1) ** 2)
    ln_L_mix = -0.5 * np.sum(np.log(2 * np.pi * err_mix**2) + ((exp_mix - teo_mix) / err_mix) ** 2)
    ln_L_3D1 = -0.5 * np.sum(np.log(2 * np.pi * err_3D1**2) + ((exp_3D1 - teo_3D1) / err_3D1) ** 2)

    # 2) Likelihood su energia di legame del deuterone
    _, mu = nnstudio.lab2rel(0, 0)
    N = 2 * nnstudio.Np
    H = np.zeros((N, N))
    T = np.zeros((N, N))

    ww = np.hstack((nnstudio.wmesh, nnstudio.wmesh))
    pp = np.hstack((nnstudio.pmesh, nnstudio.pmesh))

    V = nnstudio.setup_Vmtx(chn_3S1[0])[0]
    for i, p_bra in enumerate(pp):
        for j, p_ket in enumerate(pp):
            if i == j:
                T[i, j] = p_bra**2 / (2 * mu)
            V[i, j] = V[i, j] * p_bra * p_ket * np.sqrt(ww[i] * ww[j])

    H = T + V
    eigvals, eigvecs = linalg.eigh(H)
    srt = np.argsort(eigvals)
    E_theo = eigvals[srt[0]]
    phi_gs = eigvecs[:, srt[0]]

    # Vincolo morbido su energia e quadrupolo: evita che dominino il fit ai phase shifts
    E_obs = -2.224  # MeV
    E_err = 0.10   # MeV
    ln_L_E = -0.5 * (np.log(2 * np.pi * E_err**2) + ((E_obs - E_theo) / E_err) ** 2)

    # 3) Likelihood sul quadrupolo del deuterone
    psi_gs = phi_gs / (pp * np.sqrt(ww))
    u = psi_gs[:nnstudio.Np]
    w = psi_gs[nnstudio.Np:]

    if u[np.argmax(np.abs(u))] < 0:
        u *= -1
        w *= -1

    idx = np.argsort(nnstudio.pmesh)
    p = nnstudio.pmesh[idx]
    spline_u = CubicSpline(p, u[idx])
    spline_w = CubicSpline(p, w[idx])
    du = spline_u(p, 1)
    dw = spline_w(p, 1)
    w_sorted = w[idx]
    wmesh_sorted = nnstudio.wmesh[idx]

    SD_term = (np.sqrt(8) / 20.0) * (-p**2 * dw * du - 3.0 * p * w_sorted * du)
    DD_term = (-1.0 / 20.0) * (p**2 * dw**2 + 6.0 * w_sorted**2)
    Q_theo = np.sum(wmesh_sorted * (SD_term + DD_term)) * hbarc**2

    if Q_theo < 0:
        Q_theo *= -1

    Q_obs = 0.286  # fm^2
    Q_err = 0.03   # fm^2
    ln_L_Q = -0.5 * (np.log(2 * np.pi * Q_err**2) + ((Q_obs - Q_theo) / Q_err) ** 2)

    ln_L_tot = ln_L_3S1 + ln_L_mix + ln_L_3D1 + ln_L_E + ln_L_Q
    return ln_L_tot


def ln_posterior_NLO_3S1(parameters, nnstudio, chn_3S1, exp_3S1, err_3S1, exp_3D1, err_3D1, exp_mix, err_mix):
    ln_pr = ln_prior_NLO_3S1(parameters)

    if not np.isfinite(ln_pr):
        return -np.inf

    ln_post = ln_pr
    ln_post += ln_likelihood_NLO_3S1(parameters, nnstudio, chn_3S1, exp_3S1, err_3S1, exp_3D1, err_3D1, exp_mix, err_mix)

    return ln_post